# Modelo Eleitoral 2026 — Setup Colab Pro

Notebook receita para rodar o pipeline completo em **modo prod** no Google Colab Pro.

**Atenção**: Colab Pro tem limite de ~12h por sessão. O treino do modelo bayesiano pode extrapolar. Se isso acontecer, considere rodar em uma VM persistente (GCE, EC2) ou simplificar o pooling hierárquico.

## Antes de começar

1. Editar a célula `CONFIG` abaixo com o URL do seu repositório privado no GitHub e o `billing_project_id` do Google Cloud.
2. Ter um Personal Access Token do GitHub com escopo `repo` (para clonar repo privado).
3. Ter permissão de `BigQuery User` no projeto GCP para rodar queries via Base dos Dados.

In [ ]:
# === CONFIG — única célula que você precisa editar ===
GITHUB_REPO_URL = "https://github.com/SEU_USUARIO/eleicao2026.git"  # TROCAR
GITHUB_USER = "SEU_USUARIO"                                         # TROCAR
GCP_BILLING_PROJECT_ID = "SEU-PROJETO-GCP"                          # TROCAR

# Onde persistir dados e modelos entre sessões (Google Drive)
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/eleicao2026"
REPO_DIR = "/content/eleicao2026"

## 1. Montar Google Drive

`data/` e `models/` vão viver no Drive. Assim, se a sessão morrer, o próximo run reaproveita tudo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
os.makedirs(f"{DRIVE_PROJECT_DIR}/data", exist_ok=True)
os.makedirs(f"{DRIVE_PROJECT_DIR}/models", exist_ok=True)
print("Drive OK.")

## 2. Clonar repositório

In [ ]:
from getpass import getpass
import subprocess, os, sys

if not os.path.isdir(REPO_DIR):
    token = getpass('GitHub Personal Access Token (escopo repo): ')
    url_auth = GITHUB_REPO_URL.replace('https://', f'https://{GITHUB_USER}:{token}@')
    subprocess.run(['git', 'clone', url_auth, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print('Repo em', os.getcwd())

## 3. Instalar dependências

Colab já vem com várias libs. Instalamos o resto direto (sem criar venv — o ambiente do Colab é descartável).

In [ ]:
!pip install -q -r requirements.txt

## 4. Autenticar Google Cloud (para BigQuery)

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('GCP autenticado.')

## 5. Apontar `data/` e `models/` para o Drive (symlinks)

Assim o código do repo salva tudo no Drive automaticamente.

In [ ]:
import os, shutil

for sub in ['data', 'models']:
    local = os.path.join(REPO_DIR, sub)
    target = os.path.join(DRIVE_PROJECT_DIR, sub)
    if os.path.islink(local):
        os.unlink(local)
    elif os.path.isdir(local):
        shutil.rmtree(local)
    os.symlink(target, local)
    print(f'{local} -> {target}')

## 6. Configurar `mode: prod` e billing project

In [ ]:
import yaml
from pathlib import Path

cfg_path = Path('config.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
cfg['mode'] = 'prod'
cfg.setdefault('gcp', {})['billing_project_id'] = GCP_BILLING_PROJECT_ID
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
print('config.yaml atualizado para prod.')

## 7. Rodar o pipeline completo

Por padrão rodamos o script oficial `run_prod.sh`. Alternativamente você pode chamar cada `scripts/0X_*.py` individualmente para monitorar melhor.

In [ ]:
!bash run_prod.sh

## 8. Sanidade: verificar artefatos

Após o run, você deve ter:
- `data/raw/*.parquet` — dados ingeridos
- `data/interim/painel_mestre.parquet`
- `data/processed/features.parquet`
- `data/processed/predictions.parquet`
- `models/lgbm.pkl`, `models/bayesian.nc`
- `reports/*.md` — relatórios

In [ ]:
!ls -lh data/raw data/interim data/processed models reports 2>/dev/null || true